# 03 — MobileNetV2 Transfer Learning
This notebook trains an ImageNet-pretrained MobileNetV2. Internet access may be needed once to download torchvision pretrained weights. Run with a GPU runtime for faster training.

In [ ]:
from pathlib import Path
import sys, os, json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT/"src").exists() and (ROOT.parent/"src").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATA_DIR = ROOT/"data"/"raw"
OUTPUT_DIR = ROOT/"outputs"
for d in ["plots","confusion_matrices","metrics","predictions"]: (OUTPUT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Project root:", ROOT, "| Dataset:", DATA_DIR)

In [ ]:
import torch
from src.dataset import discover_dataset, stratified_split
from src.deep_learning_model import train_mobilenet
from src.utils import set_seed
set_seed(42)
df=discover_dataset(DATA_DIR)
train_df,val_df,test_df=stratified_split(df)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model, metrics = train_mobilenet(train_df,val_df,test_df,
    model_path=ROOT/"models"/"mobilenetv2.pth",
    metrics_path=OUTPUT_DIR/"metrics"/"mobilenetv2_metrics.json",
    epochs=15,batch_size=32,patience=4)
display(pd.DataFrame(metrics["history"]))

In [ ]:
hist=pd.DataFrame(metrics["history"])
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(hist.epoch,hist.train_loss,label="train loss")
ax.plot(hist.epoch,hist.val_loss,label="validation loss")
ax.set(xlabel="Epoch",ylabel="Cross-entropy loss",title="MobileNetV2 training curve"); ax.legend()
plt.tight_layout(); fig.savefig(OUTPUT_DIR/"plots"/"mobilenet_training_curves.png",dpi=160); plt.show()
print({k:v for k,v in metrics.items() if k!="history"})

The checkpoint stores class names, image size and normalization settings. Training and inference must use the same preprocessing. Test metrics should not be used to tune the model.